In [2]:
import sys; sys.path.append("..")      # para encontrar src/
import time
from multiprocessing import Pool, cpu_count
from pymongo import MongoClient
from pymongo.errors import BulkWriteError
from tqdm.auto import tqdm              # tqdm.auto se ve bien en notebooks

from src.kmers import procesar_lote

db = MongoClient("mongodb://localhost:27017")["panteon_prueba"]
db.kmers.create_index([("secuencia_id", 1), ("k", 1)], unique=True)
db.kmers.create_index([("te_orden", 1), ("k", 1)])

'te_orden_1_k_1'

In [3]:
ya = {d["secuencia_id"] for d in db.kmers.find({"k": 6}, {"secuencia_id": 1})}
total = db.secuencias.count_documents({})
print(f"total={total}  ya hechas={len(ya)}")

def lotes(tam=200, maximo=None):
    cursor = db.secuencias.find({}, {"te_orden": 1, "secuencia": 1})
    if maximo:
        cursor = cursor.limit(maximo)
    lote = []
    for s in cursor:
        if s["_id"] in ya:
            continue
        lote.append((s["_id"], s["te_orden"], s["secuencia"]))
        if len(lote) == tam:
            yield lote
            lote = []
    if lote:
        yield lote

total=239946  ya hechas=239946


In [4]:
def insertar(docs):
    try:
        db.kmers.insert_many(docs, ordered=False)
    except BulkWriteError as e:
        if any(err["code"] != 11000 for err in e.details["writeErrors"]):
            raise

t0 = time.perf_counter()
with Pool(max(1, cpu_count() - 1)) as pool, tqdm(total=total, initial=len(ya)) as barra:
    for docs in pool.imap(procesar_lote, lotes(maximo=1000)):   # quita maximo para todo
        insertar(docs)
        barra.update(len(docs) // 6)
print(f"{time.perf_counter() - t0:.1f} s")

100%|██████████| 239946/239946 [00:00<?, ?it/s]

2.8 s


In [5]:
# 1. Totales
print(db.kmers.count_documents({}), 6 * db.secuencias.count_documents({}))

# 2. Secuencias con ventanas descartadas (N u otros caracteres)
pipeline = [
    {"$match": {"k": 1}},
    {"$lookup": {"from": "secuencias", "localField": "secuencia_id",
                 "foreignField": "_id", "as": "s"}},
    {"$unwind": "$s"},
    {"$match": {"$expr": {"$lt": ["$total_ventanas", "$s.longitud"]}}},
    {"$count": "con_descartes"},
]
print(list(db.kmers.aggregate(pipeline)))

1439676 1439676
[{'con_descartes': 54837}]


In [12]:
from bson import ObjectId

sid = ObjectId("6ae27e4ffa276374128bf63b")

docs = list(db.kmers.find({"secuencia_id": sid}).sort("k", 1))

for d in docs:
    print(d["k"], d["total_ventanas"], len(d["conteos"]))
    print(sorted(d["conteos"].items(), key=lambda x: x[1], reverse=True))

1 1991 4
[('A', 720), ('T', 643), ('G', 331), ('C', 297)]
2 1990 16
[('AA', 306), ('TT', 234), ('AT', 229), ('TA', 193), ('GA', 123), ('TG', 113), ('AG', 110), ('TC', 103), ('CT', 102), ('CA', 97), ('GT', 78), ('GC', 75), ('AC', 74), ('GG', 55), ('CG', 53), ('CC', 45)]
3 1989 64
[('AAA', 144), ('TTT', 97), ('AAT', 90), ('ATT', 74), ('TAT', 70), ('ATA', 68), ('TAA', 66), ('TTA', 65), ('GAA', 50), ('ATG', 47), ('CAA', 45), ('AAG', 45), ('AGA', 45), ('TGA', 44), ('GAT', 43), ('ATC', 40), ('CTT', 39), ('TTC', 39), ('TCA', 34), ('TCT', 34), ('TTG', 33), ('CTA', 30), ('GTA', 30), ('TAG', 29), ('ACT', 28), ('TGT', 28), ('TAC', 28), ('GCT', 27), ('AAC', 26), ('CAT', 26), ('TGC', 25), ('GTT', 24), ('ACA', 23), ('GCA', 23), ('AGG', 22), ('AGT', 22), ('AGC', 21), ('CTG', 20), ('CAG', 19), ('GGA', 18), ('TCG', 18), ('GAG', 17), ('TCC', 17), ('CCA', 17), ('TGG', 16), ('CGA', 16), ('GGC', 15), ('GCG', 14), ('CGT', 14), ('ACG', 14), ('CGC', 14), ('GGT', 14), ('CTC', 13), ('GAC', 13), ('GTG', 13), ('C